In [1]:
from transformers import pipeline
import spacy
import pandas as pd
import gradio as gr
from sklearn.linear_model import LinearRegression

import warnings
warnings.filterwarnings('ignore')

In [2]:
# Load pre-trained Hugging Face model for text classification
classifier = pipeline('zero-shot-classification', model="facebook/bart-large-mnli")

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

In [3]:
def classify_prompt(prompt):
    labels = ["descriptive_statistics", "correlation", "regression", "others"]
    result = classifier(prompt, candidate_labels=labels)
    return result['labels'][0]  # Return the highest scored label

In [4]:
nlp = spacy.load("en_core_web_sm")

def extract_variables(prompt):
    doc = nlp(prompt)
    variables = [ent.text for ent in doc.ents if ent.label_ == "GPE" or ent.label_ == "ORG"]  # Customize for variables
    if len(variables) == 0:
        return None
    return variables

In [5]:
def load_data(file):
    if not file.name.endswith('.csv'):
        raise ValueError("Please upload a CSV file.")
    return pd.read_csv(file.name)  # Use file.name to read the file

def get_columns(df):
    return df.columns.tolist()

In [6]:
def descriptive_statistics(df, variables):
    return df[variables].describe()

def correlation(df, variables):
    return df[variables].corr()

def regression(df, dependent_var, independent_vars):
    X = df[independent_vars]
    y = df[dependent_var]
    model = LinearRegression()
    model.fit(X, y)
    return model.coef_, model.intercept_

In [7]:
def analyze_data(file, prompt):
    # Step 1: Classify the prompt
    analysis_type = classify_prompt(prompt)

    # Step 2: Load the data
    try:
        df = load_data(file)
    except ValueError as e:
        return str(e)

    columns = get_columns(df)

    # Step 3: Extract variables from the prompt
    variables = extract_variables(prompt)

    # Fallback: detect column names mentioned in the prompt
    if not variables:
        prompt_lower = prompt.lower()
        variables = [col for col in columns if col.lower() in prompt_lower]

    if not variables:
        return f"Please specify the variables for analysis. Available variables: {', '.join(columns)}"

    # Ensure variables is a list
    if isinstance(variables, str):
        variables = [variables]

    # Validate variables
    if not set(variables).issubset(columns):
        missing = [v for v in variables if v not in columns]
        return f"Some variables not found in the dataset: {', '.join(missing)}"

    # Step 4: Perform analysis
    if analysis_type == "descriptive_statistics":
        result = descriptive_statistics(df, variables)
    elif analysis_type == "correlation":
        result = correlation(df, variables)
    elif analysis_type == "regression":
        if len(variables) < 2:
            return "Regression requires one dependent variable and at least one independent variable."
        result = regression(df, variables[0], variables[1:])
    else:
        return "Analysis type not recognized."

    return result

In [ ]:
# Create Gradio Interface
interface = gr.Interface(fn=analyze_data,
                         inputs=[gr.File(label="Upload CSV File"), gr.Textbox(label="Analysis Prompt")],
                         outputs="text",
                         title = "Gen AI Data Analysis Assistant")

interface.launch(debug = True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
